In [6]:
import pandas as pd

permits = pd.read_csv("permits_standardized.csv")

print("SF proposed_use / sub_type values:")
print(permits[permits["city"]=="San Francisco"]["sub_type"].value_counts().head(15))

print("\nSJ sub_type values:")
print(permits[permits["city"]=="San Jose"]["sub_type"].value_counts().head(15))

print("\nOakland sub_type values:")
print(permits[permits["city"].str.contains("Oakland")]["sub_type"].value_counts().head(15))

SF proposed_use / sub_type values:
sub_type
Multi-Family            1072
Other                    829
Single Family            808
Office/Medical           123
Industrial/Warehouse      53
Parking/Garage            53
Retail/Commercial         44
Civic/Institutional       41
Hotel/Motel               18
Unknown                   14
ADU/2nd Unit               1
Name: count, dtype: int64

SJ sub_type values:
sub_type
ADU/2nd Unit            3925
Single Family           3366
Multi-Family            1255
Retail/Commercial       1086
Office/Medical           648
Unknown                  492
Condo                    371
Other                    210
Industrial/Warehouse     147
Mixed Use                126
Parking/Garage            69
Civic/Institutional       62
Hotel/Motel               32
Name: count, dtype: int64

Oakland sub_type values:
sub_type
Single Family           821
Other                   308
Multi-Family            273
Retail/Commercial       127
Parking/Garage          114
Con

# Adding New Permit Columns & Replacing Old Ones

In [7]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — Load and inspect all three datasets
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

train = pd.read_csv("training_final_v2.csv")
val   = pd.read_csv("validation_final_v2.csv")
test  = pd.read_csv("testing_final_v2.csv")

for df in [train, val, test]:
    df["tract_id"] = df["tract_id"].astype(str).str.zfill(11)

print("=== TRAINING ===")
print(f"Shape: {train.shape}")
print(f"Columns: {list(train.columns)}")
dup = train["tract_id"].value_counts()
print(f"Tracts appearing once: {(dup==1).sum()}, twice: {(dup==2).sum()}")

print("\n=== VALIDATION ===")
print(f"Shape: {val.shape}")
dup_v = val["tract_id"].value_counts()
print(f"Tracts appearing once: {(dup_v==1).sum()}, twice: {(dup_v==2).sum()}")

print("\n=== TESTING ===")
print(f"Shape: {test.shape}")

# Find the period/year column in training
candidate_cols = [c for c in train.columns if any(k in c.lower() for k in
                  ["year", "period", "time", "wave", "obs"])]
print(f"\nCandidate period columns in training: {candidate_cols}")
for c in candidate_cols:
    print(f"  {c}: {sorted(train[c].unique())}")

# Show an example duplicate tract
example = dup[dup==2].index[0]
print(f"\nExample tract {example} — both rows:")
print(train[train["tract_id"]==example][candidate_cols + ["tract_id"]].to_string())

=== TRAINING ===
Shape: (1744, 45)
Columns: ['tract_id', 'population', 'median_income', 'median_rent', 'median_home_value', 'share_college', 'share_25_34', 'homeownership_rate', 'vacancy_rate', 'share_pre1940', 'share_white', 'share_black', 'share_hispanic', 'poverty_rate', 'rent_burden', 'jobs_arts', 'jobs_food', 'jobs_total', 'arts_density', 'food_density', 'gentrify_density', 'arts_job_share', 'food_job_share', 'area_sqmi', 'year', 'city_sf', 'city_oakland', 'city_san_jose', 'city', 'jan_housing_value_begin', 'jan_housing_value_end', 'city_median_income', 'gentrified', 'below_city_median_income', 'criteria_met', 'new_const_count_train', 'new_const_valuation_sum_train', 'new_const_sqft_sum_train', 'new_const_units_sum_train', 'all_permits_count_train', 'city_group', 'avg_valuation_per_permit', 'avg_sqft_per_permit', 'avg_units_per_permit', 'any_permit_activity']
Tracts appearing once: 0, twice: 872

=== VALIDATION ===
Shape: (872, 45)
Tracts appearing once: 872, twice: 0

=== TESTING

In [9]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — Load permits, fix tract_id, map buckets, build aggregates
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

permits = pd.read_csv("permits_standardized.csv")

# Drop permits that failed geocoding (no tract_id)
n_before = len(permits)
permits = permits.dropna(subset=["tract_id"])
print(f"Dropped {n_before - len(permits):,} permits with missing tract_id ({len(permits):,} remaining)")

# Fix: float tract_ids like 6085504322.0 → "06085504322"
permits["tract_id"] = (
    permits["tract_id"]
    .astype(float).astype(int).astype(str)
    .str.zfill(11)
)

if "year_filed" not in permits.columns:
    permits["year_filed"] = pd.to_datetime(permits["permit_date"], errors="coerce").dt.year

SUBTYPE_MAP = {
    "Single Family":        "single_family",
    "Multi-Family":         "multifamily",
    "Condo":                "multifamily",
    "ADU/2nd Unit":         "multifamily",
    "Retail/Commercial":    "commercial",
    "Office/Medical":       "commercial",
    "Hotel/Motel":          "commercial",
    "Mixed Use":            "commercial",
    "Industrial/Warehouse": "industrial",
    "Parking/Garage":       "industrial",
    "Civic/Institutional":  "civic_other",
    "Other":                "civic_other",
    "Unknown":              "civic_other",
}
permits["bucket"] = permits["sub_type"].map(SUBTYPE_MAP).fillna("civic_other")

BUCKETS = ["single_family", "multifamily", "commercial", "industrial", "civic_other"]

def aggregate_buckets(permits_df, year_start, year_end):
    window    = permits_df[permits_df["year_filed"].between(year_start, year_end)]
    new_const = window[window["permit_type"] == "New Construction"]

    bucket_counts = (
        new_const.groupby(["tract_id", "bucket"])
        .size().unstack(fill_value=0).reset_index()
    )
    for b in BUCKETS:
        if b not in bucket_counts.columns:
            bucket_counts[b] = 0

    total = window.groupby("tract_id").size().reset_index(name="total_permit_count")

    agg = bucket_counts[["tract_id"] + BUCKETS].merge(total, on="tract_id", how="outer")
    agg[BUCKETS + ["total_permit_count"]] = agg[BUCKETS + ["total_permit_count"]].fillna(0).astype(int)
    return agg

permits_0610 = aggregate_buckets(permits, 2006, 2010)
permits_1115 = aggregate_buckets(permits, 2011, 2015)
permits_1620 = aggregate_buckets(permits, 2016, 2020)
permits_2124 = aggregate_buckets(permits, 2021, 2024)

for label, df in [("2006-2010", permits_0610), ("2011-2015", permits_1115),
                  ("2016-2020", permits_1620), ("2021-2024", permits_2124)]:
    print(f"{label}: {len(df):,} tracts | sample IDs: {df['tract_id'].head(3).tolist()}")

# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — Drop old permit columns, merge new ones by period
# ════════════════════════════════════════════════════════════════════════════

OLD_KEYWORDS = [
    "new_const", "all_permits", "avg_valuation", "avg_sqft", "avg_units",
    "any_permit", "city_group", "sqft", "valuation",
]
PERMIT_COLS = BUCKETS + ["total_permit_count"]

def drop_old(df):
    to_drop = [c for c in df.columns if any(k in c for k in OLD_KEYWORDS)]
    print(f"  Dropping {len(to_drop)} old permit columns: {to_drop}")
    return df.drop(columns=to_drop, errors="ignore")

def attach_permits(df_slice, permits_agg):
    merged = df_slice.merge(permits_agg, on="tract_id", how="left")
    merged[PERMIT_COLS] = merged[PERMIT_COLS].fillna(0).astype(int)
    return merged

# Training — route each row to its correct permit window
train = pd.read_csv("training_final_v2.csv")
train["tract_id"] = train["tract_id"].astype(str).str.zfill(11)
print("Training:"); train = drop_old(train)

rows_2010 = attach_permits(train[train["year"] == 2010].copy(), permits_0610)
rows_2015 = attach_permits(train[train["year"] == 2015].copy(), permits_1115)
train_out = pd.concat([rows_2010, rows_2015]).sort_index()

print(f"  year=2010: {len(rows_2010)} rows | permit coverage: {(rows_2010['total_permit_count'] > 0).sum()}")
print(f"  year=2015: {len(rows_2015)} rows | permit coverage: {(rows_2015['total_permit_count'] > 0).sum()}")

# Validation — single window 2016-2020
val = pd.read_csv("validation_final_v2.csv")
val["tract_id"] = val["tract_id"].astype(str).str.zfill(11)
print("\nValidation:"); val = drop_old(val)
val_out = attach_permits(val, permits_1620)
print(f"  {len(val_out)} rows | permit coverage: {(val_out['total_permit_count'] > 0).sum()}")

# Testing — single window 2021-2024
test = pd.read_csv("testing_final_v2.csv")
test["tract_id"] = test["tract_id"].astype(str).str.zfill(11)
print("\nTesting:"); test = drop_old(test)
test_out = attach_permits(test, permits_2124)
print(f"  {len(test_out)} rows | permit coverage: {(test_out['total_permit_count'] > 0).sum()}")


# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — Separate named-city vs Other tracts
# ════════════════════════════════════════════════════════════════════════════

def split_city(df, label):
    named = df[df["city"] != "Other"].copy()
    other = df[df["city"] == "Other"].copy()
    print(f"{label}: {len(named)} named-city rows, {len(other)} Other rows")
    return named, other

train_named, train_other = split_city(train_out, "Training")
val_named,   val_other   = split_city(val_out,   "Validation")
test_named,  test_other  = split_city(test_out,  "Testing")


# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — Save all outputs
# ════════════════════════════════════════════════════════════════════════════

train_named.to_csv("training_named_cities.csv",  index=False)
train_other.to_csv("training_other_tracts.csv",  index=False)
val_named.to_csv("validation_named_cities.csv",  index=False)
val_other.to_csv("validation_other_tracts.csv",  index=False)
test_named.to_csv("testing_named_cities.csv",    index=False)
test_other.to_csv("testing_other_tracts.csv",    index=False)

print("Saved all 6 files.")
print(f"\nFinal permit features: {PERMIT_COLS}")
print("\nNamed-city shapes:")
print(f"  Training   : {train_named.shape}")
print(f"  Validation : {val_named.shape}")
print(f"  Testing    : {test_named.shape}")
print("\nOther-tract shapes:")
print(f"  Training   : {train_other.shape}")
print(f"  Validation : {val_other.shape}")
print(f"  Testing    : {test_other.shape}")

# Permit feature summary for named-city training
print("\nPermit feature summary (training named cities):")
print(train_named[PERMIT_COLS].describe().round(1))

Dropped 3,468 permits with missing tract_id (13,260 remaining)
2006-2010: 364 tracts | sample IDs: ['06001400100', '06001400200', '06001400300']
2011-2015: 345 tracts | sample IDs: ['06001400100', '06001400200', '06001400300']
2016-2020: 422 tracts | sample IDs: ['06001400100', '06001400200', '06001400300']
2021-2024: 338 tracts | sample IDs: ['06001400500', '06001400800', '06001401000']
Training:
  Dropping 10 old permit columns: ['new_const_count_train', 'new_const_valuation_sum_train', 'new_const_sqft_sum_train', 'new_const_units_sum_train', 'all_permits_count_train', 'city_group', 'avg_valuation_per_permit', 'avg_sqft_per_permit', 'avg_units_per_permit', 'any_permit_activity']
  year=2010: 872 rows | permit coverage: 250
  year=2015: 872 rows | permit coverage: 233

Validation:
  Dropping 10 old permit columns: ['new_const_count_val', 'new_const_valuation_sum_val', 'new_const_sqft_sum_val', 'new_const_units_sum_val', 'all_permits_count_val', 'city_group', 'avg_valuation_per_permit'

# Download the completed files

In [10]:
import google.colab.files as files

file_names = [
    "training_named_cities.csv",
    "training_other_tracts.csv",
    "validation_named_cities.csv",
    "validation_other_tracts.csv",
    "testing_named_cities.csv",
    "testing_other_tracts.csv"
]

for file_name in file_names:
    print(f"Downloading {file_name}...")
    files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>